In [1]:
# ==========================================
# Hydrophobicity CLKD Datapoint-Wise (7 classes)
# ==========================================

import sys
print(sys.executable)

import numpy as np
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.models import load_model, clone_model
from tensorflow.keras.losses import CategoricalCrossentropy, KLDivergence
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

/home/22EC1102/miniconda3/envs/tinyml_env/bin/python


2026-04-30 19:16:48.752443: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777556808.794786 1517960 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777556808.809960 1517960 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777556808.845037 1517960 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777556808.845079 1517960 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777556808.845083 1517960 computation_placer.cc:177] computation placer alr

In [2]:
# CELL 2: GPU + mixed precision (critical order)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    tf.config.set_logical_device_configuration(
        gpus[0],
        [tf.config.LogicalDeviceConfiguration(memory_limit=9000)]  # 4GB limit
    )
    print("✅ GPU memory growth + 4GB limit set!")



✅ GPU memory growth + 4GB limit set!


In [3]:
# ==========================================
# CELL 4: MIXED PRECISION
# ==========================================
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy("mixed_float16")
print("Mixed precision enabled.")

Mixed precision enabled.


In [4]:
# ==========================================
# Load Teacher & Student Models
# ==========================================
# Teacher: MobileNetV3Large (224x224, hydrophobicity 7-class)
teacher = keras.models.load_model("/home/22EC1102/soumen/satarupa/hydrophobicity/teacher_model_mobilenetv3.keras")
teacher.trainable = False

# Student: 32x32 hydrophobicity student
original_student_model = keras.models.load_model("/home/22EC1102/soumen/satarupa/hydrophobicity/hydro_kd_M1.keras")

print("Teacher input shape :", teacher.input_shape)
print("Teacher output shape:", teacher.output_shape)
print("Student input shape :", original_student_model.input_shape)
print("Student output shape:", original_student_model.output_shape)

I0000 00:00:1777556815.799240 1517960 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9000 MB memory:  -> device: 0, name: Tesla V100-PCIE-32GB, pci bus id: 0000:37:00.0, compute capability: 7.0
I0000 00:00:1777556815.870772 1517960 cuda_executor.cc:479] failed to allocate 8.79GiB (9437184000 bytes) from device: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
I0000 00:00:1777556815.871362 1517960 cuda_executor.cc:479] failed to allocate 7.91GiB (8493465600 bytes) from device: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
I0000 00:00:1777556815.871940 1517960 cuda_executor.cc:479] failed to allocate 7.12GiB (7644119040 bytes) from device: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
I0000 00:00:1777556815.872510 1517960 cuda_executor.cc:479] failed to allocate 6.41GiB (6879707136 bytes) from device: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
I0000 00:00:1777556815.873184 1517960 cuda_e

Teacher input shape : (None, 224, 224, 3)
Teacher output shape: (None, 7)
Student input shape : (None, 32, 32, 3)
Student output shape: (None, 7)


/home/22EC1102/miniconda3/envs/tinyml_env/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 44 variables whereas the saved optimizer has 4 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/home/22EC1102/miniconda3/envs/tinyml_env/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 40 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [5]:
# ==========================================
# Clone Student (keep original unchanged)
# ==========================================
student_model = keras.models.clone_model(original_student_model)
#student_model.set_weights(original_student_model.get_weights())

print("Cloned student input shape :", student_model.input_shape)
print("Cloned student output shape:", student_model.output_shape)

Cloned student input shape : (None, 32, 32, 3)
Cloned student output shape: (None, 7)


In [6]:
# ==========================================
# Load Hydrophobicity Confidence Regions (224x224)
# ==========================================
low_conf_data = np.load('low_confidence_data.npy', mmap_mode="r")
low_conf_labels = np.load('low_confidence_labels.npy', mmap_mode="r")
medium_conf_data = np.load('medium_confidence_data.npy', mmap_mode="r")
medium_conf_labels = np.load('medium_confidence_labels.npy', mmap_mode="r")
high_conf_data = np.load('high_confidence_data.npy', mmap_mode="r")
high_conf_labels = np.load('high_confidence_labels.npy', mmap_mode="r")

print("Low confidence data shape :", low_conf_data.shape)
print("Low confidence labels shape :", low_conf_labels.shape)
print("Medium confidence data shape :", medium_conf_data.shape)
print("Medium confidence labels shape :", medium_conf_labels.shape)
print("High confidence data shape :", high_conf_data.shape)
print("High confidence labels shape :", high_conf_labels.shape)

Low confidence data shape : (934, 224, 224, 3)
Low confidence labels shape : (934, 7)
Medium confidence data shape : (933, 224, 224, 3)
Medium confidence labels shape : (933, 7)
High confidence data shape : (933, 224, 224, 3)
High confidence labels shape : (933, 7)


In [7]:
# ==========================================
# Preprocess Data (MobileNetV3 style)
# ==========================================
# Normalize [0,255] → preprocess_input expects [0,255] → [-1,1]
low_conf_data = preprocess_input(low_conf_data.astype('float32'))
medium_conf_data = preprocess_input(medium_conf_data.astype('float32'))
high_conf_data = preprocess_input(high_conf_data.astype('float32'))

# Labels already one-hot (7 classes)
print("Low data range after preprocess:", low_conf_data.min(), low_conf_data.max())

Low data range after preprocess: 0.0 255.0


In [8]:
# Optional normalization check
print("Low data range   :", low_conf_data.min(), low_conf_data.max())
print("Medium data range:", medium_conf_data.min(), medium_conf_data.max())
print("High data range  :", high_conf_data.min(), high_conf_data.max())

Low data range   : 0.0 255.0
Medium data range: 0.0 255.0
High data range  : 0.0 255.0


In [9]:
# ==========================================
# Distillation Loss (7 classes)
# ==========================================
def distillation_loss(y_true, student_logits, teacher_logits, alpha, temperature):
    cce = CategoricalCrossentropy()
    ce_loss = cce(y_true, student_logits)

    student_soft = tf.nn.softmax(student_logits / temperature, axis=1)
    teacher_soft = tf.nn.softmax(teacher_logits / temperature, axis=1)

    kd_loss = KLDivergence()(teacher_soft, student_soft) * (temperature ** 2)
    total_loss = alpha * kd_loss + (1.0 - alpha) * ce_loss
    return total_loss

In [10]:
# ==========================================
# Train on Confidence Region
# ==========================================
def train_on_region(
    x_train, y_train, alpha, temperature, student_model, teacher_model,
    batch_size=16, epochs=30, lr=1e-4, shuffle_buffer=2000
):
    optimizer = Adam(learning_rate=lr)
    best_acc = -1.0
    best_weights = None
    
    dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
    dataset = dataset.shuffle(min(len(x_train), shuffle_buffer), seed=SEED)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        epoch_acc = 0.0
        batches = 0
        
        for x_batch, y_batch in dataset:
            x_teacher = x_batch
            x_student = tf.image.resize(x_batch, [32, 32])
            
            with tf.GradientTape() as tape:
                student_logits = student_model(x_student, training=True)
                teacher_logits = teacher_model(x_teacher, training=False)
                loss = distillation_loss(y_batch, student_logits, teacher_logits, alpha, temperature)
            
            grads = tape.gradient(loss, student_model.trainable_variables)
            optimizer.apply_gradients(zip(grads, student_model.trainable_variables))
            
            preds = tf.argmax(student_logits, axis=1)
            true_labels = tf.argmax(y_batch, axis=1)
            batch_acc = tf.reduce_mean(tf.cast(tf.equal(preds, true_labels), tf.float32))
            
            epoch_loss += float(loss.numpy())
            epoch_acc += float(batch_acc.numpy())
            batches += 1
        
        epoch_loss /= batches
        epoch_acc /= batches
        
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")
        
        # CRITICAL FIX: Save best weights IMMEDIATELY
        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_weights = student_model.get_weights()
            student_model.set_weights(best_weights)
            print(f"  → NEW BEST: {best_acc:.4f}")
    
    print(f"Region complete - Best Accuracy: {best_acc:.4f}")
    return best_weights, best_acc

In [11]:
# ==========================================
# Hydrophobicity Alpha/Tuning (from your alpha code)
# ==========================================
# Use your calculated alpha values
alpha_low, T_low = 0.2051, 5     # small model low Cd
alpha_med, T_med = 0.2024, 5     # medium model med Cd  
alpha_high, T_high = 0.2016, 5   # large model high Cd

print("alpha_low, T_low :", alpha_low, T_low)
print("alpha_med, T_med :", alpha_med, T_med)
print("alpha_high, T_high :", alpha_high, T_high)

alpha_low, T_low : 0.2051 5
alpha_med, T_med : 0.2024 5
alpha_high, T_high : 0.2016 5


In [12]:
print(high_conf_data.shape)

(933, 224, 224, 3)


In [ ]:
# ==========================================
# Train High Confidence Region LAST
# ==========================================
print("\n=== HIGH CONFIDENCE REGION (easiest) ===")
best_weights_high, best_acc_high, losses_high, accs_high = train_on_region(
    x_train=high_conf_data,
    y_train=high_conf_labels,
    alpha=alpha_high,
    temperature=T_high,
    student_model=student_model,
    teacher_model=teacher,
    batch_size=32,
    epochs=10,
    lr=1e-4
)

print(f"High confidence region done - Best Accuracy: {best_acc_high:.4f}")


=== HIGH CONFIDENCE REGION (easiest) ===


2026-04-30 19:17:31.315491: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 561770496 exceeds 10% of free system memory.
2026-04-30 19:17:32.549065: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 561770496 exceeds 10% of free system memory.
2026-04-30 19:17:35.966557: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 561770496 exceeds 10% of free system memory.
I0000 00:00:1777556856.871180 1517960 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-30 19:18:40.816408: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 1/10 - Loss: 1.5646 - Acc: 0.1552
  → NEW BEST: 0.1552


2026-04-30 19:19:08.953759: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 2/10 - Loss: 1.5633 - Acc: 0.1615
  → NEW BEST: 0.1615
Epoch 3/10 - Loss: 1.5628 - Acc: 0.1635
  → NEW BEST: 0.1635


2026-04-30 19:20:03.591643: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 4/10 - Loss: 1.5599 - Acc: 0.1733
  → NEW BEST: 0.1733
Epoch 5/10 - Loss: 1.5568 - Acc: 0.2015
  → NEW BEST: 0.2015
Epoch 6/10 - Loss: 1.5522 - Acc: 0.2363
  → NEW BEST: 0.2363


In [ ]:
# ==========================================
# Train Medium Confidence Region
# ==========================================
print("\n=== MEDIUM CONFIDENCE REGION ===")
best_weights_med, best_acc_med, losses_med, accs_med = train_on_region(
    x_train=medium_conf_data,
    y_train=medium_conf_labels,
    alpha=alpha_med,
    temperature=T_med,
    student_model=student_model,
    teacher_model=teacher,
    batch_size=32,
    epochs=10,
    lr=1e-4
)

print(f"Medium confidence region done - Best Accuracy: {best_acc_med:.4f}")


=== MEDIUM CONFIDENCE REGION ===
Epoch 1/10 - Loss: 0.7364 - Accuracy: 0.6333
Epoch 2/10 - Loss: 0.7098 - Accuracy: 0.6256
Epoch 3/10 - Loss: 0.7105 - Accuracy: 0.6263
Epoch 4/10 - Loss: 0.6904 - Accuracy: 0.6398
Epoch 5/10 - Loss: 0.6364 - Accuracy: 0.6731


2026-04-29 19:52:52.588580: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 6/10 - Loss: 0.6520 - Accuracy: 0.6446
Epoch 7/10 - Loss: 0.7151 - Accuracy: 0.6450
Epoch 8/10 - Loss: 0.6319 - Accuracy: 0.6683
Epoch 9/10 - Loss: 0.6587 - Accuracy: 0.6508
Epoch 10/10 - Loss: 0.6575 - Accuracy: 0.6408
Medium confidence region done - Best Accuracy: 0.6731


In [ ]:
# ==========================================
# Train Low Confidence Region FIRST
# ==========================================
print("=== LOW CONFIDENCE REGION (hardest) ===")
best_weights_low, best_acc_low, losses_low, accs_low = train_on_region(
    x_train=low_conf_data,
    y_train=low_conf_labels,
    alpha=alpha_low,
    temperature=T_low,
    student_model=student_model,
    teacher_model=teacher,
    batch_size=32,    # Smaller batch for stability
    epochs=10,
    lr=1e-4
)

print(f"Low confidence region done - Best Accuracy: {best_acc_low:.4f}")

=== LOW CONFIDENCE REGION (hardest) ===


Epoch 1/10 - Loss: 0.8629 - Accuracy: 0.5750
Epoch 2/10 - Loss: 0.7476 - Accuracy: 0.5976
Epoch 3/10 - Loss: 0.6928 - Accuracy: 0.6434
Epoch 4/10 - Loss: 0.6791 - Accuracy: 0.6528
Epoch 5/10 - Loss: 0.7094 - Accuracy: 0.6309
Epoch 6/10 - Loss: 0.6422 - Accuracy: 0.6715
Epoch 7/10 - Loss: 0.6438 - Accuracy: 0.6719
Epoch 8/10 - Loss: 0.6556 - Accuracy: 0.6441
Epoch 9/10 - Loss: 0.7029 - Accuracy: 0.6392
Epoch 10/10 - Loss: 0.6413 - Accuracy: 0.6597
Low confidence region done - Best Accuracy: 0.6719


In [ ]:
# ==========================================
# Save Final Curriculum Student
# ==========================================
student_model.save("hydro_curriculum_M1.keras")
print("Hydrophobicity curriculum student saved as 'hydro_curriculum_M1.keras'")

Hydrophobicity curriculum student saved as 'hydro_curriculum_M1.keras'


In [ ]:
# ==========================================
# Verify Final Model
# ==========================================
print("Final student input shape :", student_model.input_shape)
print("Final student output shape:", student_model.output_shape)
print("Num classes:", student_model.output_shape[-1])  # Should be 7

Final student input shape : (None, 32, 32, 3)
Final student output shape: (None, 7)
Num classes: 7


In [ ]:
# ==========================================
# Hydrophobicity Validation Evaluation
# ==========================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

# ==========================================
# Load Final Curriculum Model
# ==========================================
final_model = keras.models.load_model("hydro_curriculum_M1.keras")
final_model.compile(
    optimizer=Adam(),
    loss=CategoricalCrossentropy(),
    metrics=["accuracy"]
)

print("Final model loaded - input:", final_model.input_shape)
print("Output classes:", final_model.output_shape[-1])  # 7

Final model loaded - input: (None, 32, 32, 3)
Output classes: 7


In [ ]:
# ==========================================
# Hydrophobicity Validation Data (32x32 for student)
# ==========================================
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

val_data_student32 = val_datagen.flow_from_directory(
    "/home/22EC1102/soumen/satarupa/hydrophobicity/Hydrophobicity Classes Photos/validation",
    target_size=(32, 32),  # Student size
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

print("Validation batches:", len(val_data_student32))
print("Validation classes:", val_data_student32.class_indices)

Found 700 images belonging to 7 classes.
Validation batches: 22
Validation classes: {'HC1': 0, 'HC2': 1, 'HC3': 2, 'HC4': 3, 'HC5': 4, 'HC6': 5, 'HC7': 6}


In [ ]:
# ==========================================
# Evaluate Curriculum Student on Validation
# ==========================================
print("=== Hydrophobicity Validation Results ===")
val_loss, val_acc = final_model.evaluate(
    val_data_student32,
    verbose=1
)

print(f"Hydrophobicity Val Loss: {val_loss:.4f}")
print(f"Hydrophobicity Val Acc:  {val_acc:.4f} ({val_acc*100:.2f}%)")

=== Hydrophobicity Validation Results ===


I0000 00:00:1777472808.270723 1199746 service.cc:152] XLA service 0x7f7b9c005280 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777472808.270770 1199746 service.cc:160]   StreamExecutor device (0): Tesla V100-PCIE-32GB, Compute Capability 7.0
2026-04-29 19:56:48.309223: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777472821.817975 1199746 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


22/22 ━━━━━━━━━━━━━━━━━━━━ 29s 696ms/step - accuracy: 0.6886 - loss: 0.6924
Hydrophobicity Val Loss: 0.6924
Hydrophobicity Val Acc:  0.6886 (68.86%)
